# Vraksh 
A library consumes documents and answer questions based on that document

## Canvas Designer Library

#### Console Print Class

In [2]:
## This takes care of standard output
from rich.console import Console
from rich.table import Table
from rich.pretty import Pretty

console =  Console()

def table_create(table_name, *columns):
    table = Table(title=table_name, show_header= True)

    for col in columns:
        table.add_column(col)
    
    return table


def table_print(table):
    console.print(table)

# Base Parser Class

In [3]:
class Parser:

    def __init__(self):
        pass
    
    def parse(self):
        print("Override this method to parse the scene")
        
    def print(self):
        print("Override this method to print the results")
    
    def graph(self, scene):
        displacy.render(scene, style="dep", jupyter=True)
        

import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_trf")

/Users/aaggarwal/Documents/personal/projects/vraksh/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/aaggarwal/Documents/personal/projects/vraksh/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Action Parser

In [4]:
# VR-4565
class ActionParser(Parser):

    def get_action_list(self, scene):

        result = []
        for token in scene:
            if token.pos_.casefold() == "VERB".casefold():
                result.append(token)
        
        return result

    def get_main(self, scene):
        for token in scene:
            if token.dep_.casefold() == "ROOT".casefold():
                return token
        
        return None

    def get_action_by_text(self, scene, token_text):
        for token in scene:
            if token.text.casefold() == token_text.casefold():
                return token
        
        return None

    
    def is_main(self, action):
        return action.dep_.casefold() == "ROOT".casefold()

    def is_aux(self, action):
        return action.dep_.casefold() == "aux".casefold()

    def is_present(self, action):
        return action.tag_.casefold() == "VBZ".casefold() or action.tag_.casefold() == "VBP".casefold()

    def is_past(self, action):
        return action.tag_.casefold() == "VBD".casefold() or action.tag_.casefold() == "VBN".casefold()

    def is_future(self, action):
        return action.tag_.casefold() == "MD".casefold()
        


### Testing 
scene = nlp("Sunday, I eat fruits and drink water at 12:00 midnight")
action_parser = ActionParser()
action_list = action_parser.get_action_list(scene)

assert len(action_list) == 2
assert action_parser.get_main(scene).text == "eat"
assert action_parser.get_action_by_text(scene, "eat").text == "eat"

token_eat = action_parser.get_action_by_text(scene, "eat")
assert action_parser.is_present(token_eat) == True

#### Action Receiver Parser

In [5]:
# VR-1212
class ActionReceiverParser(Parser):


    def get_action_receiver(self, token_action):
    
        result = []
    
        for token in token_action.children:

            ## Now check if this action has any direct object which is non living
            if (token.dep_.casefold() == "dobj"):
                result.append(token)

            ## We have to handle the cases when object is not direct object and living object
            if (token.dep_.casefold() == "prep"):
                result.extend(self._parse_prep(token))


        return result


    # In this method we will evaluate if object could be the prep object
    def _parse_prep(self, token):

        result = []

        if (token.text.casefold() == "with") or (token.text.casefold() == "to"):

            prep_obj_list = [child for child in token.children if child.dep_.casefold() == "pobj"]
            if list(prep_obj_list):
                result.append(prep_obj_list[0])

        return result


### Testing 

scene = nlp("I am playing with my calculator which is placed on the table")
action_parser = ActionParser()
action_token_playing = action_parser.get_action_by_text(scene, "playing")

action_receiver_parser = ActionReceiverParser()
result = action_receiver_parser.get_action_receiver(action_token_playing)
assert result[0].text == "calculator" 

scene = nlp("I am going to school")
action_parser = ActionParser()
action_token_going = action_parser.get_action_by_text(scene, "going")

action_receiver_parser = ActionReceiverParser()
result = action_receiver_parser.get_action_receiver(action_token_going)
assert result[0].text == "school"

scene = nlp("I am playing cricket in the ground")
action_parser = ActionParser()
action_token_playing = action_parser.get_action_by_text(scene, "playing")

action_receiver_parser = ActionReceiverParser()
result = action_receiver_parser.get_action_receiver(action_token_playing)
assert result[0].text == "cricket", "FAILED" 

#### Action Doer Designer Class

In [6]:
# VR-4323
class ActionDoerParser(Parser):


    def get_action_doer(self, token_verb):

        result = []

        for token in token_verb.children:

            if token.dep_.casefold() == "nsubj".casefold():
                result.append(token)

        return result

    def get_action_doer_by_text(self, token_verb, token_text):
        result = []

        for token in token_verb.children:

            if token.dep_.casefold() == "nsubj".casefold() and token.text.casefold() == token_text.casefold():
                result.append(token)

        return result
    

    def is_person(self, token):
        return token.ent_type_ == "PERSON" 

    def is_gpe(self, token):
        return token.ent_type_ == "GPE"

    def is_organization(self, token):
        return token.ent_type_ == "ORG" 
        
    def is_location(self, token):
        return token.ent_type_ == "LOC"

    def is_facility(self, token):
        return token.ent_type_ == "FAC"

    def is_product(self, token):
        return token.ent_type_ == "PRODUCT"

    def is_event(self, token):
        return token.ent_type_ == "EVENT"

    def is_work_of_art(self, token):
        return token.ent_type_ == "WORK_OF_ART"

    def is_law(self, token):
        return token.ent_type_ == "LAW"

    def is_language(self, token):
        return token.ent_type_ == "LANGUAGE"




### Testing 

scene = nlp("Amit eats fruits")
action_doer_parser = ActionDoerParser()
action_parser = ActionParser()
action_eat = action_parser.get_action_by_text(scene, "eats")
action_doer_amit = action_doer_parser.get_action_doer_by_text(action_eat, "Amit")[0]
assert action_doer_parser.is_person(action_doer_amit) == True

scene = nlp("India does good job")
action_doer_parser = ActionDoerParser()
action_parser = ActionParser()
action_does = action_parser.get_action_by_text(scene, "does")
action_doer_india = action_doer_parser.get_action_doer_by_text(action_does, "India")[0]
assert action_doer_parser.is_gpe(action_doer_india) == True




#### Action Temporal Designer

In [7]:
import dateparser
import pendulum

# VR-3434
class ActionTemporalParser(Parser):


    def get_action_time(self, token_verb):

        temp_result = []
        self.result = ""

        ## Parse the verb's children to extract day and time information
        for child in token_verb.children:

            if child.dep_.casefold() == "npadvmod":
                temp_result.append("".join(tok.text_with_ws for tok in child.subtree).strip())

            if child.dep_.casefold() == "prep" and child.text.casefold() in {"at", "around", "on"}:                
                temp_result.append("".join(tok.text_with_ws for tok in child.subtree).strip())

        self.result = " ".join(temp_result)
        return self.result
    

    def print(self):
        print(self.result)
    


### Testing 

scene = nlp("Tomorrow , I eat fruits at 12:00 AM")

action_parser = ActionParser()
action_eat = action_parser.get_action_by_text(scene, "eat")

action_time_parser = ActionTemporalParser()
action_time_parser.get_action_time(action_eat)
action_time_parser.print()
action_time_parser.graph(scene)


Tomorrow at 12:00 AM


In [8]:
import dateparser
import pendulum 

scene1 = nlp("Sunday, Ram eats fruits at 12:00 AM")
action_parser = ActionParser() 
action_eat = action_parser.get_main(scene1)
assert action_eat.text == "eats", "FAILED"

action_temporal_parser = ActionTemporalParser()
action_eat_time = action_temporal_parser.get_action_time(action_eat)
action_eat_time_dt = dateparser.parse(action_eat_time)
action_eat_time_pendulum = pendulum.instance(action_eat_time_dt)


scene2 = nlp("Monday, Ram drinks juice at 1:00 AM")
action_parser = ActionParser() 
action_drink = action_parser.get_main(scene2)

action_temporal_parser = ActionTemporalParser()
action_drink_time = action_temporal_parser.get_action_time(action_drink)
action_drink_time_dt = dateparser.parse(action_drink_time)
action_drink_time_pendulum = pendulum.instance(action_drink_time_dt)


assert action_eat_time_pendulum < action_drink_time_pendulum, "FAILED"







### Scene Class

In [ ]:
from rich import print

import spacy
from spacy import displacy
from spacy.tokens import Token


class Scene:

    def __init__(self, scene_str):
        self.scene = nlp(scene_str)
        Token.set_extension("_to_element", method=self._to_element, force=True)
    
    
    def get_action(self):
        action_parser = ActionParser(self.scene)
        return action_parser.parse()


    def get_action_doer(self, action):
        action_doer_parser = ActionDoerParser(action)
        return action_doer_parser.parse()

    def get_action_time(self, action):
        action_time_parser = ActionTimeParser(action)
        return action_time_parser.parse()


    def get_action_receiver(self, action, action_doer):
        action_receiver_parser = ActionReceiverParser(action, action_doer)
        return action_receiver_parser.parse()

    




## Timeline Class

In [ ]:
from uuid import UUID

from rich import print
from tinydb import TinyDB, Query
from rich import print_json
import json


class Timeline:

    def __init__(self):
        self.db = TinyDB('my_db.json')
        self.id = 0
        self.db.truncate()

    def get_unique_id(self):
        self.id = self.id + 1
        return self.id

    
    def add_action(self, unique_id, action: Element):

        query = Query()
        result = self.db.search(query.id == unique_id)
        if not result:
            self.db.insert({"id" : unique_id, "action" : action._to_dict()})
        else:
            query = Query()
            self.db.update({"action" : action._to_dict()}, query.id == unique_id)



    def add_action_doer(self, unique_id, action :Element, action_doer :Element):

        query = Query()
        result = self.db.search(query.id == unique_id)
        if not result:
            self.db.insert({"id" : unique_id, "action" : action._to_dict(), "action_doer" : action_doer._to_dict()})
        else:
            self.db.update({"action_doer" : action_doer._to_dict()}, query.id == unique_id)


    def add_action_time(self, unique_id, action :Element, action_time :Element):

        query = Query()
        result = self.db.search(query.id == unique_id)
        if not result:
            if isinstance(action_time, list):
                self.db.insert({"id" : unique_id, "action" : action._to_dict(), "action_time" : [ x._to_dict() for x in action_time ]})
            else:
                self.db.insert({"id" : unique_id, "action" : action._to_dict(), "action_time" :  action_time._to_dict()})
        else:
            query = Query()
            if isinstance(action_time, list):
                self.db.update({"id" : unique_id, "action_time" : [ x._to_dict() for x in action_time ]}, query.id == unique_id)
            else:
                self.db.update({"id" : unique_id, "action_time" :  action_time._to_dict()}, query.id == unique_id)



    def add_action_receiver(self, unique_id, action :Element, action_doer:Element, action_receiver :Element):

        query = Query()
        result = self.db.search(query.id == unique_id)
        if not result:
            self.db.insert({"id" : unique_id, "action" : action._to_dict(), "action_receiver" : action_receiver._to_dict()})
        else:
            query = Query()
            self.db.update({"action_receiver" : action_receiver._to_dict()}, query.id == unique_id)

            

    def add_action_doer_location(self, unique_id, action, action_doer, location):
        pass


    def add_action_object_location(self, unique_id, action , action_object, location):
        pass


    def clear(self):
        self.db.truncate()


    def render(self):
        pretty_json = json.dumps(self.db.all(), indent=4)
        print(pretty_json)


    def query(self, query):

        if isinstance(query, QuerySubject):
            return self.query_subject(query)


    def query_subject(self, query_subject):

        ## Find subject which are matching action name 
        query = Query()
        result = self.db.search(query.action.text == query_subject.action.text)
        return result



In [150]:
class QuerySubject:

    def action(self, action):
        self.action = action
        return self

    # action_time = None, action_location = None, action_receiver = None):
    def action_time(self, action_time):
        self.action_time = action_time
        return self

    def action_location(self, action_location):
        self.action_location = action_location
        return self

    def action_receiver(self, action_receiver):
        self.action_receiver = action_receiver
        return self

## Canvas Reader Library

### Canvas Reader Class


# Main

In [158]:
from rich import print

import spacy
from spacy import displacy
from spacy.tokens import Token

nlp = spacy.load("en_core_web_trf")

timeline = Timeline()

scene = Scene("Amit wake up at 09:00 AM today")
scene.add_scene_to_timeline(timeline)

scene2 = Scene("Rahul drank water at 10:00 AM")
scene2.add_scene_to_timeline(timeline)

scene3 = Scene("Deepak took bath at 10:30 AM")
scene3.add_scene_to_timeline(timeline)

scene4 = Scene("Praveen started my office at 11:00 AM")
scene4.add_scene_to_timeline(timeline)

# timeline.render()

# ### ----------------------------------------------------------

scene_reader = Scene("who took bath at 10:30 AM")


query_subject = QuerySubject()
query_subject.action(scene_reader.get_action()[0])

result = timeline.query(query=query_subject)

for r in result:
    print(r)


TypeError: Object of type Element is not JSON serializable